In [ ]:
%pwd            

'c:\\Users\\Asus\\Machine_learning\\LLM\\Language_Model\\GPT_from_scratch\\notebook'

In [3]:
import os 

In [4]:
os.chdir("..\.")

In [5]:
%pwd

'c:\\Users\\Asus\\Machine_learning\\LLM\\Language_Model\\GPT_from_scratch'

In [ ]:
import torch 
import torch.nn as nn 
from torch.nn import functional as F

from tqdm import tqdm




batch_size  = 32
block_size  = 8
device      = "cuda" if torch.cuda.is_available() else "cpu"
eval_interval   = 300
learning_rate   = 1e-3 
max_iters       = 3000
eval_iter       = 200
n_emd           = 32 


torch.manual_seed(1337)
with open ("data\gpt_train.txt","r") as file:
    text    = file.read()


# All the unique characters that occur in this text 
chars       = sorted((set(text)))
vocab_size  = len(chars)

## creating mapping from characters to integers 
stoi    = {ch:i for i,ch in enumerate(chars)}
#itos    = {i:ch for i,ch in enumerate(chars)}
itos    = dict(enumerate(chars))
encode  = lambda word: [stoi[i] for i in word]
decode  = lambda integers: "".join(itos[int(i)] for i in integers)



## Let's encode entire dataset of file. 
data = torch.tensor(encode(text),dtype=torch.long,device="cuda")

## Lets split the data into train and val dataset 
n   = int(0.9 * len(data))
train_data  = data[:n]
val_data    = data[n:]


def get_batch(split):
    data    = train_data if split == "train" else val_data
    ix      = torch.randint(len(data)-block_size,(4,))
    x       = torch.stack([data[i   : i+block_size] for i in ix]).to(device="cuda:0")
    y       = torch.stack([data[i+1 : i+block_size+1] for i in ix]).to(device="cuda:0")
    return x,y 

In [18]:
vocab_size,block_size,n_emd

(65, 8, 32)

In [51]:
class BiGramLanguageModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.token_embedding    = nn.EmbeddingBag(vocab_size,n_emd)
        self.linear_embedding   = nn.Linear(n_emd, vocab_size)

In [17]:
        
## ------------------------------------------------------------------------ ## 
class BiGramLanguageModel(nn.Module):

    def __init__(self):
        super().__init__()
        # each token directly reads off the logits for next token from a lookup table 
        self.token_embedding_table      = nn.Embedding(vocab_size,n_emd)
        self.position_embedding_table   = nn.Embedding(block_size,n_emd)
        self.lm_head                    = nn.Linear(n_emd,vocab_size)       
        

    def forward(self,idx,target=None):
        B,T         = idx.shape 
        # index and target are both (B,T) tensor of integers
        tok_emb     = self.token_embedding_table(idx)   # its arrange in the shape of ==================================================================>> (B,T,C)
        pos_emb     = self.position_embedding_table(torch.arange(T,device=device)) # This embdding gives the idea of where word is belong in a sentence=>> (T,C) 
        x           = tok_emb + pos_emb                 # combined representation of token and its position.=========================>> (B,T,C) + (T,C) == (B,T,C) 
        logits      = self.lm_head(tok_emb)             # for getting token_emb to logits we need linear layer,shape ===================================>> (B,T,vocab_size) 

        if target is None:
            loss    = None
        else:    
            B,T,C   = logits.shape
            logits  = logits.view(B*T,C)    # cross entropy input expectation is (minibatch,C)
            target  = target.view(B*T)      
            loss    = nn.functional.cross_entropy(logits,target)
        return logits,loss
    
    def generate(self,idx,max_new_tokens):
        # idx is (B,T) array of indices in the current context. 
        for _ in range(max_new_tokens):
            logits,loss = self.forward(idx)                         # (B,T,C)
            logits      = logits[:,-1,:]                            # (B,C)
            probs       = nn.functional.softmax(logits,dim=-1)      # (B,C)
            # sample from the distribution 
            idx_next    = torch.multinomial(probs,num_samples=1)    # (B,1)
            # append sample index to running sequence 
            idx         = torch.cat([idx,idx_next],dim=1)           # (B,T+1)
        return idx
    


- Consider the 'Idx', we going to encoded them based on identity of the tokens inside the 'idx'. 

- 🧠 1. tok_emb — Token Embeddings

    - These embeddings capture what each token (word, subword, or character) represents.
    - For example, the token "the" might map to a vector like [0.1, 0.8, ..., -0.2].

But by themselves, token embeddings don’t include any info about where the token appears in a sentence.
- 📍 2. pos_emb — Positional Embeddings

    - These embeddings encode where a token appears in the sequence (i.e., position 0, 1, 2, ...).
    - Position 0 (first token) has a vector like [0.2, -0.1, ..., 0.3], position 1 a different one, etc.

So even if "the" appears twice in a sentence, at different positions, pos_emb helps the model treat each occurrence differently based on context and order.

- **?????? Why added them together??** 
    -  Its representing the combined representation, that contains
        - The identity of the token (its semantics).
        - Its position in the sequence (its role or function based on context).

1. Token Embedding

- Token embedding is the process of converting individual "tokens"  into dense numerical vectors. 
- Are designed to capture the semantic meaning and contextual relationships of the tokens.

**Why it's needed:**

- Computers don't understand text directly. They operate on numbers. Token embeddings provide a way to represent words (or parts of words) in a numerical format that machine learning models can process. 
- The "embedding space" is designed so that words with similar meanings (e.g., "king" and "queen," or "cat" and "kitten") have similar vector representations (i.e., their vectors are close to each other in this high-dimensional space).

#### How it works
- Tokenization:  Raw text is broken down into smaller units called tokens. This can be words ("hello", "world"), subwords ("run", "##ning" for "running"), or characters.

- Vocabulary: A vocabulary of all unique tokens in the training data is created. Each unique token is assigned a unique integer ID.

    - Embedding Layer: An embedding layer (often implemented as a lookup table or a learnable matrix) is then used. When a token's ID is fed into this layer, it outputs its corresponding dense vector representation.

    - Learning: These embedding vectors are not fixed; they are learned during the training of the neural network. Through backpropagation, the model adjusts the values in these vectors so that they effectively capture semantic and syntactic properties relevant to the task (e.g., predicting the next word, translating, classifying sentiment).

- Example:

    - "cat" might be embedded as [0.2, -0.5, 0.8, ...]
    - "dog" might be embedded as [0.3, -0.4, 0.7, ...] (similar to "cat" as they are both animals)
    - "table" might be embedded as [-0.9, 0.1, 0.3, ...] (very different from "cat")

2. Position Embedding (or Positional Encoding)

What it is:
Position embedding is a mechanism used primarily in Transformer models to inject information about the relative or absolute position of tokens within a sequence. Unlike recurrent neural networks (RNNs) that process words sequentially and inherently capture order, Transformers process all tokens in a sequence in parallel. This parallel processing means they lose the inherent sense of word order. Positional embeddings solve this problem.

- Why it's needed:
The order of words is crucial for understanding meaning. Consider:

    - "Man bites dog."
    - "Dog bites man."
- The words are the same, but their positions completely change the meaning. Without positional information, a Transformer would see these as just bags of words.

In [7]:
## ------------------------------------------------------------------------ ## 
model       = BiGramLanguageModel().to(device="cuda")
optimizer   = torch.optim.AdamW(model.parameters(),lr=learning_rate)

c:\Users\Asus\anaconda3\envs\llm_env\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [8]:
@torch.no_grad()
def estimate_loss():
    out = {}
    model.eval()
    for split in ["train","val"]:
        losses = torch.zeros(eval_iter)
        for k in range(eval_iter):
            X,Y         = get_batch("train")
            logits,loss = model(X,Y)
            losses[k]   = loss.item()   
        out[split] = losses.mean()
    model.train()
    return out

In [9]:
for iter in range(max_iters):
    if iter % eval_interval == 0:
        losses  = estimate_loss()
        print(f"step {iter}: train loss {losses['train']:.4f},val loss {losses['val']:.4f}")

    #sample a batch of data
    xb,yb = get_batch("train")
    #evaluate the loss
    logits,loss = model(xb,yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()


context = torch.zeros((1,1),dtype=torch.long,device="cuda:0")
print(decode(model.generate(context,max_new_tokens=500)[0]))

step 0: train loss 4.3338,val loss 4.3451
step 300: train loss 3.2115,val loss 3.1930
step 600: train loss 2.8528,val loss 2.8550
step 900: train loss 2.7075,val loss 2.7183
step 1200: train loss 2.6900,val loss 2.6384
step 1500: train loss 2.6273,val loss 2.6682
step 1800: train loss 2.5962,val loss 2.6248
step 2100: train loss 2.5891,val loss 2.5936
step 2400: train loss 2.6069,val loss 2.5714
step 2700: train loss 2.5571,val loss 2.5568


RuntimeError: CUDA error: device-side assert triggered
CUDA kernel errors might be asynchronously reported at some other API call, so the stacktrace below might be incorrect.
For debugging consider passing CUDA_LAUNCH_BLOCKING=1.
Compile with `TORCH_USE_CUDA_DSA` to enable device-side assertions.
